# Bases de données — Exercice 3 : les stations Vélib' (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue : la requête SQL à taper dans la **console H2**, et le résultat obtenu |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** `SELECT ... WHERE`, `COUNT`, `AVG`, `LIKE`, `MIN`/`MAX` et requête imbriquée, `AND`, `ORDER BY` (`DESC`), jointure entre deux relations par une clé étrangère.

> ⚠️ **Liberté prise :** le fichier `velib.sql` du professeur n'est pas disponible en ligne. Le fichier `velib.sql` placé à côté de ce notebook a été **reconstitué** à partir des **données ouvertes** officielles de Vélib' Métropole (flux GBFS public), avec exactement le schéma relationnel du sujet. C'est un instantané pris le **14 novembre 2026 à 2 h 18** (en pleine nuit) : les nombres obtenus ici seront **différents** de ceux de la classe, car les données changent toutes les minutes. **Les requêtes, elles, sont les mêmes.**

> Ces données obéissent au schéma relationnel ci-dessous.
> - Information(<u>id</u> : CHAR, name : CHAR, lat : REAL, lon : REAL, capacity : INT)
> - Status(id\* : CHAR, is_installed : INT, is_renting : INT, is_returning : INT, last_reported : CHAR, num_bikes_available : INT, num_bikes_available_ebike : INT, num_bikes_available_mechanical : INT, num_docks_available : INT)
> - La clé étrangère id de la relation Status fait référence à la clé primaire id de la relation Information.
>
> **Information :** `id` identifiant de la station ; `name` nom ; `capacity` nombre de bornettes ; `lat`, `lon` latitude et longitude.
> **Status :** `is_installed` station déployée (1/0) ; `is_renting` peut louer des vélos (1/0) ; `is_returning` peut recevoir des vélos (1/0) ; `last_reported` date de la dernière mise à jour ; `num_bikes_available` vélos disponibles (dont `..._ebike` électriques et `..._mechanical` mécaniques) ; `num_docks_available` bornettes disponibles.
>
> **Consignes.** Télécharger et exécuter le contenu du fichier velib.sql. Écrire des requêtes SQL **uniques** pour afficher les informations ci-dessous.

---
# MISE EN PLACE

En classe, les requêtes SQL sont tapées dans la **console H2** (le SGBD utilisé dans le cours). C'est aussi le cas ici : **chaque requête de ce notebook a été exécutée dans H2**, et le résultat affiché sous la requête est celui qu'a donné H2. Il n'y a **aucun code à exécuter** dans ce notebook : il suffit de recopier les requêtes dans la console H2.

**Pour ouvrir la console H2 :** lancer « H2 Console », se connecter avec la JDBC URL `jdbc:h2:~/nsi`, l'utilisateur `sa` et un mot de passe vide.

> ⚠️ **Liberté prise :** les résultats ont été obtenus avec H2 version 2.5.252 et recopiés tels quels, à trois détails près : la durée d'exécution affichée par H2 (« 12 ms ») a été supprimée, car elle change à chaque fois ; pour les résultats longs, seules les 12 premières lignes sont recopiées, suivies de « ... » et du nombre total de lignes ; dans les messages d'erreur, le nom technique de l'erreur Java (`org.h2.jdbc....Exception`) a été remplacé par « Erreur : » ; quand plusieurs instructions sont exécutées d'un coup, chacune est rappelée au-dessus de son résultat. Dans la console du navigateur, le résultat apparaît sous forme de tableau ; le contenu est le même.

**Chargement des données** (le fichier `velib.sql` contient les `CREATE TABLE` et les `INSERT INTO`) :

```sql
RUNSCRIPT FROM '/chemin/vers/velib.sql';
SELECT * FROM Information;
SELECT * FROM Status;
```

*Résultat dans la console H2 :*

```text
RUNSCRIPT FROM '/chemin/vers/velib.sql';
(Update count: 3049)

SELECT * FROM Information;
ID          | NAME                                               | LAT       | LON       | CAPACITY
213688169   | Benjamin Godard - Victor Hugo                      | 48.865982 | 2.275725  | 35
19179944124 | Hôpital Mondor                                     | 48.798923 | 2.4537451 | 28
18795078746 | Charcot - Benfleet                                 | 48.87837  | 2.4405239 | 28
36255       | Toudouze - Clauzel                                 | 48.879295 | 2.3373601 | 21
251039991   | Cassini - Denfert-Rochereau                        | 48.837524 | 2.3360355 | 25
85002689    | Jourdan - Stade Charléty                           | 48.819427 | 2.3433354 | 60
2515829865  | Basilique                                          | 48.936268 | 2.3588667 | 22
516709288   | Charonne - Robert et Sonia Delaunay                | 48.855907 | 2.3925707 | 20
37874517    | Saint-Sulpice                                      | 48.851654 | 2.3308077 | 21
210565680   | Vaneau - Sèvres                                    | 48.848564 | 2.320422  | 35
210561800   | Saint-Romain - Cherche-Midi                        | 48.84708  | 2.321375  | 17
209063434   | André Karman - République                          | 48.9104   | 2.3851357 | 31
...
(1522 rows)

SELECT * FROM Status;
ID          | IS_INSTALLED | IS_RENTING | IS_RETURNING | LAST_REPORTED       | NUM_BIKES_AVAILABLE | NUM_BIKES_AVAILABLE_EBIKE | NUM_BIKES_AVAILABLE_MECHANICAL | NUM_DOCKS_AVAILABLE
213688169   | 1            | 1          | 1            | 2026-11-14 02:00:52 | 17                  | 14                        | 3                              | 17
19179944124 | 1            | 1          | 1            | 2026-11-14 02:05:05 | 16                  | 4                         | 12                             | 12
18795078746 | 1            | 1          | 1            | 2026-11-14 02:00:32 | 10                  | 9                         | 1                              | 18
36255       | 1            | 1          | 1            | 2026-11-14 02:07:47 | 6                   | 5                         | 1                              | 15
251039991   | 1            | 1          | 1            | 2026-11-14 02:06:09 | 3                   | 3                         | 0                              | 20
85002689    | 1            | 1          | 1            | 2026-11-14 02:05:52 | 33                  | 18                        | 15                             | 26
2515829865  | 1            | 1          | 1            | 2026-11-14 02:02:05 | 8                   | 6                         | 2                              | 13
516709288   | 1            | 1          | 1            | 2026-11-14 02:07:15 | 7                   | 4                         | 3                              | 12
37874517    | 1            | 1          | 1            | 2026-11-14 02:07:14 | 11                  | 1                         | 10                             | 10
210565680   | 1            | 1          | 1            | 2026-11-14 02:07:02 | 28                  | 6                         | 22                             | 5
210561800   | 1            | 1          | 1            | 2026-11-14 02:07:07 | 15                  | 4                         | 11                             | 2
209063434   | 1            | 1          | 1            | 2026-11-14 02:04:03 | 12                  | 10                        | 2                              | 18
...
(1522 rows)
```

---
# QUESTION 1

#### 📝 ÉNONCÉ

> 1. Le nombre total de stations.

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Information;
```

*Résultat dans la console H2 :*

```text
COUNT(*)
1522
(1 row)
```

**Réponse :** 1 522 stations (avec cet instantané).

#### 💡 EXPLICATION PÉDAGOGIQUE

- Chaque station a **un** enregistrement dans `Information` (son `id` est la clé primaire) : compter les enregistrements, c'est compter les stations.
- `SELECT COUNT(*) FROM Status;` donnerait ici le même nombre, mais c'est `Information` qui décrit les stations.

---
# QUESTION 2

#### 📝 ÉNONCÉ

> 2. Le nombre moyen de vélos électriques disponibles.

#### ✅ RÉSOLUTION

```sql
SELECT AVG(num_bikes_available_ebike) FROM Status;
```

*Résultat dans la console H2 :*

```text
AVG(NUM_BIKES_AVAILABLE_EBIKE)
5.030880420499343
(1 row)
```

**Réponse :** environ 5,03 vélos électriques par station.

#### 💡 EXPLICATION PÉDAGOGIQUE

- `AVG` (*average*) calcule la **moyenne** d'un attribut sur tous les enregistrements sélectionnés.
- Le nombre de vélos électriques disponibles est un attribut de la relation `Status` (l'état de la station à un instant donné), et non de `Information` (les caractéristiques fixes de la station).

---
# QUESTION 3

#### 📝 ÉNONCÉ

> 3. Les noms des stations dont le nom contient le mot Lecourbe.

#### ✅ RÉSOLUTION

```sql
SELECT name FROM Information WHERE name LIKE '%Lecourbe%';
```

*Résultat dans la console H2 :*

```text
NAME
Lecourbe - Jean Maridor
Charles Lecocq - Lecourbe
Sèvres - Lecourbe
Lecourbe - Convention
Volontaires - Lecourbe
(5 rows)
```

**Réponse :** 5 stations : Lecourbe - Jean Maridor, Charles Lecocq - Lecourbe, Sèvres - Lecourbe, Lecourbe - Convention, Volontaires - Lecourbe.

#### 💡 EXPLICATION PÉDAGOGIQUE

- « **contient** le mot » → `LIKE` avec `%` des deux côtés : `%` remplace **n'importe quelle suite de caractères** (même vide). `'%Lecourbe%'` trouve donc *Lecourbe* au début, au milieu ou à la fin du nom.
- ⚠️ `WHERE name = 'Lecourbe'` ne trouverait **rien** : aucune station ne s'appelle exactement « Lecourbe ».
- ⚠️ `'Lecourbe%'` ne trouverait que les noms qui **commencent** par Lecourbe (2 stations sur 5).
- Dans H2, `LIKE` distingue les majuscules : il faut écrire `Lecourbe` avec un L majuscule.

---
# QUESTION 4

#### 📝 ÉNONCÉ

> 4. Le nom de la station située la plus au sud.

#### ✅ RÉSOLUTION

```sql
SELECT name FROM Information
    WHERE lat = (SELECT MIN(lat) FROM Information);
```

*Résultat dans la console H2 :*

```text
NAME
Adrien Raynal - Verger
(1 row)
```

**Réponse :** *Adrien Raynal - Verger* (avec ces données).

#### 💡 EXPLICATION PÉDAGOGIQUE

- La plus au sud = la **plus petite latitude** `lat`.
- Le sujet demande une requête **unique** : on ne peut pas calculer `MIN(lat)` puis recopier la valeur à la main. On utilise une **requête imbriquée** (exemple page 8 du polycopié) : `(SELECT MIN(lat) FROM Information)` est calculée d'abord, puis utilisée comme une valeur dans le `WHERE`.
- ⚠️ `SELECT name, MIN(lat) FROM Information;` est **refusée** par H2 : on ne peut pas mélanger un attribut (une valeur par ligne) et une fonction d'agrégation (une seule valeur).

---
# QUESTION 5

#### 📝 ÉNONCÉ

> 5. Les noms et nombre de vélos disponibles des stations dont le nom contient le mot Lecourbe.

#### ✅ RÉSOLUTION

```sql
SELECT name, num_bikes_available FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE name LIKE '%Lecourbe%';
```

*Résultat dans la console H2 :*

```text
NAME                      | NUM_BIKES_AVAILABLE
Lecourbe - Jean Maridor   | 15
Charles Lecocq - Lecourbe | 24
Sèvres - Lecourbe         | 45
Lecourbe - Convention     | 28
Volontaires - Lecourbe    | 30
(5 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- Le **nom** est dans `Information`, le **nombre de vélos** est dans `Status` : il faut une **jointure**. C'est l'exemple même du cours (page 7).
- La condition de jointure `Information.id = Status.id` relie chaque station à son état : `Status.id` est une **clé étrangère** qui fait référence à la clé primaire `Information.id`.
- Ici, les deux relations ont un attribut `id` : dans la condition de jointure, le préfixe `Information.` / `Status.` est **obligatoire**. Pour `name` et `num_bikes_available`, qui n'existent que dans une seule relation, il est facultatif.

---
# QUESTION 6

#### 📝 ÉNONCÉ

> 6. Le nombre de stations vides qui sont déployées et peuvent recevoir des vélos.

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Status
    WHERE num_bikes_available = 0 AND is_installed = 1 AND is_returning = 1;
```

*Résultat dans la console H2 :*

```text
COUNT(*)
36
(1 row)
```

**Réponse :** 36 stations (avec cet instantané).

#### 💡 EXPLICATION PÉDAGOGIQUE

- On traduit chaque mot de la question en une condition :
  - « vides » → aucun vélo disponible : `num_bikes_available = 0` ;
  - « déployées » → `is_installed = 1` ;
  - « peuvent recevoir des vélos » → `is_returning = 1`.
- Les trois conditions doivent être vraies **en même temps** : on les relie par `AND`.
- Tous ces attributs sont dans `Status` : pas besoin de jointure, puisqu'on ne demande que le nombre.

---
# QUESTION 7

#### 📝 ÉNONCÉ

> 7. Le nom de la station avec le plus grand nombre de vélos disponibles.

#### ✅ RÉSOLUTION

```sql
SELECT name FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_bikes_available = (SELECT MAX(num_bikes_available) FROM Status);
```

*Résultat dans la console H2 :*

```text
NAME
Route de Sèvres - Porte de Bagatelle
(1 row)
```

**Réponse :** *Route de Sèvres - Porte de Bagatelle*, avec 65 vélos (avec cet instantané).

#### 💡 EXPLICATION PÉDAGOGIQUE

- C'est la combinaison des questions 4 et 5 : une **requête imbriquée** pour le maximum, et une **jointure** parce que le nom est dans `Information` alors que le nombre de vélos est dans `Status`.
- Si plusieurs stations avaient le même nombre maximal de vélos, elles seraient **toutes** affichées : la requête est juste dans tous les cas.

---
# QUESTION 8

#### 📝 ÉNONCÉ

> 8. Les noms et coordonnées GPS des stations pleines, dans l'ordre alphabétique.

#### ✅ RÉSOLUTION

```sql
SELECT name, lat, lon FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_docks_available = 0
    ORDER BY name;
```

*Résultat dans la console H2 :*

```text
NAME                                          | LAT       | LON
Argenson - Château                            | 48.888557 | 2.2642002
Aristide Briand - Place de la Résistance      | 48.821243 | 2.2511003
Charlot - Stade Gabriel Voisin                | 48.82723  | 2.2681224
Convention - Lourmel                          | 48.842056 | 2.2859912
Crozatier - Faubourg Saint-Antoine            | 48.85044  | 2.3789015
Danielle Casanova - Ledru-Rollin              | 48.815784 | 2.3832598
Enghien - Faubourg Poissonnière               | 48.87242  | 2.3483953
Faidherbe - Dahomey                           | 48.851357 | 2.3838232
Guersant - Gouvion-Saint-Cyr                  | 48.882877 | 2.2876673
Gustave Eiffel - Parc des Impressionnistes    | 48.903908 | 2.2946908
Haut du Mont-Mesly                            | 48.780193 | 2.4685738
Jules Vallès - Charonne                       | 48.854286 | 2.3847675
...
(37 rows)
```

**Réponse :** 37 stations pleines (avec cet instantané).

#### 💡 EXPLICATION PÉDAGOGIQUE

- Une station est **pleine** quand on ne peut plus y rendre de vélo : il ne reste **aucune bornette libre**, `num_docks_available = 0`.
- ⚠️ « Pleine » ne se traduit pas par `num_bikes_available = capacity` : certaines bornettes peuvent être hors service, et la question 9 montre que le nombre de vélos peut même **dépasser** la capacité.
- Coordonnées GPS = `lat` et `lon` ; « ordre alphabétique » = `ORDER BY name`, placé **à la fin** de la requête.

---
# QUESTION 9

#### 📝 ÉNONCÉ

> 9. Les noms, nombres de vélos disponibles et capacités des stations dont le nombre de vélos disponibles est supérieur à la capacité, triés dans l'ordre décroissant de leurs capacités.

#### ✅ RÉSOLUTION

```sql
SELECT name, num_bikes_available, capacity FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_bikes_available > capacity
    ORDER BY capacity DESC;
```

*Résultat dans la console H2 :*

```text
NAME                                   | NUM_BIKES_AVAILABLE | CAPACITY
BNF - Bibliothèque Nationale de France | 50                  | 42
Westermeyer - Paul Vaillant-Couturier  | 31                  | 25
Sebastopol - Rambuteau                 | 18                  | 16
(3 rows)
```

**Réponse :** 3 stations avec cet instantané : *BNF - Bibliothèque Nationale de France* (50 vélos pour 42 bornettes), *Westermeyer - Paul Vaillant-Couturier* (31 pour 25), *Sebastopol - Rambuteau* (18 pour 16).

#### 💡 EXPLICATION PÉDAGOGIQUE

- Dans le `WHERE`, on peut **comparer deux attributs** entre eux : `num_bikes_available > capacity` (le premier est dans `Status`, le second dans `Information`, d'où la jointure).
- `ORDER BY capacity DESC` : tri par capacité, **décroissant** (`DESC`, *descending*). Sans `DESC`, le tri serait croissant.
- **Comment est-ce possible ?** Plus de vélos que de bornettes ! Certaines stations acceptent des vélos « en surplus » (rattachés à la station sans être accrochés à une bornette), ou les données ne sont pas parfaitement à jour. Une base de données reflète le monde réel, avec ses imperfections : c'est justement ce que cette question fait découvrir.